# Guardrails (LangChain 1.3+)

**Guardrails** validate and filter content at key points in the agent loop — so unsafe inputs, leaked PII, and risky tool calls never reach production unchecked.

They are implemented as **middleware** on `create_agent(..., middleware=[...])` (see [6_middleware.ipynb](6_middleware.ipynb) for hook details).

**This notebook covers:**

- Deterministic vs model-based guardrails
- Built-in: `PIIMiddleware`, `HumanInTheLoopMiddleware`
- Custom: `before_agent` (input filter) and `after_agent` (output safety)
- Layered / combined guardrails
- Real-world pattern: healthcare chatbot

**Prerequisite:** `OPENAI_API_KEY` in repo-root `.env`. Prior: [6_middleware.ipynb](6_middleware.ipynb).

**Sections:** 1 overview | 2 approaches | 3 PII | 4 human-in-the-loop | 5–6 custom hooks | 7 layered stack | 8 healthcare | 9 summary

## 1. What are guardrails?

Guardrails intercept execution **before**, **during**, and **after** the agent runs.

Training reference: **[reference/guardrails-flow.png](reference/guardrails-flow.png)**

![Guardrails flow — middleware checkpoints from user input to safe response](reference/guardrails-flow.png)


### Common use cases

| Use case | Example |
|----------|---------|
| PII leakage prevention | Redact emails / cards before logging or sending to model |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests early |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure responses meet safety standards |

| Hook | Guardrail examples |
|------|-------------------|
| `before_agent` | Keyword blocklist, auth check, rate limit |
| `before_model` | PII redaction on input |
| `wrap_tool_call` | Human approval, tool sandboxing |
| `after_agent` | Output safety judge, disclaimer injection |

## 2. Two approaches: deterministic vs model-based

| | **Deterministic** | **Model-based** |
|---|-------------------|-----------------|
| How | Regex, keywords, rules | LLM / classifier judges content |
| Pros | Fast, cheap, predictable | Catches nuanced violations |
| Cons | Misses paraphrases | Slower, costs tokens |
| Best for | First line of defense | Semantic safety checks |

**Best practice:** run **deterministic checks first**, then model-based only when needed.

In [32]:
"""Load API keys from .env."""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

value = os.getenv("OPENAI_API_KEY")
if value:
    os.environ["OPENAI_API_KEY"] = value
else:
    print("OPENAI_API_KEY: missing (add to .env)")

In [33]:
import re

# --- Deterministic: keyword blocklist ---

def deterministic_guardrail(text: str) -> bool:
    """Return True if content should be blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)


test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail ===
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads


In [34]:
from langchain_openai import ChatOpenAI

# --- Model-based: LLM safety judge ---

def model_based_guardrail(text: str) -> str:
    """Returns SAFE or UNSAFE from a lightweight model."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = (
        "Is the following user input safe to process? "
        "Reply with only 'SAFE' or 'UNSAFE'.\n\n"
        f"Input: {text}"
    )
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()


print("=== Model-Based Guardrail ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads


## 3. Built-in: `PIIMiddleware`

Detects and handles **Personally Identifiable Information** before/after model calls.

### Supported PII types

| Type | Example |
|------|---------|
| `email` | user@example.com |
| `credit_card` | 5105-1051-0510-5100 |
| `ip` | 192.168.1.1 |
| `mac_address` | 00:1A:2B:3C:4D:5E |
| `url` | https://secret-site.com |
| `api_key` | Custom regex detector |

### Strategies

| Strategy | Result |
|----------|--------|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-5100` |
| `hash` | `a8f5f167...` |
| `block` | Raises exception — stops execution |

In [35]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


pii_agent = create_agent(
    model="gpt-4o-mini",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)
print("PII agent created (graph preview disabled — avoids Mermaid PNG in Jupyter).")

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [31]:
# Email + card are redacted/masked before the model sees them
result = pii_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is 5105-1051-0510-5100. "
            "Can you help me?"
        ),
    }]
})

print("=== Agent response ===")
print(result["messages"][-1].content)
print()
print("=== First message (redacted input) ===")
print(result["messages"][0].content)

=== Agent response ===
I found your customer record associated with your email. How can I assist you further?

=== First message (redacted input) ===
My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?


In [ ]:
# API keys trigger block strategy — raises before model runs
try:
    pii_agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456",
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

## 4. Built-in: `HumanInTheLoopMiddleware`

Pauses execution **before** sensitive tools run so a human can approve, edit, or reject.

**Best for:** financial transactions, external emails, production deletes.

**Requires:** `checkpointer=InMemorySaver()` + stable `thread_id`. Full approve/reject/edit flows: [6_middleware.ipynb §8](6_middleware.ipynb).

| Config | Meaning |
|--------|---------|
| `interrupt_on={"send_email": True}` | Pause before this tool |
| `interrupt_on={"search_web": False}` | Auto-run, no pause |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"


hitl_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        ),
    ],
    checkpointer=InMemorySaver(),
)
print("Human-in-the-loop agent created.")

In [ ]:
# Approve flow — include subject/body so the model calls send_email (triggers interrupt)
config_approve = {"configurable": {"thread_id": "guardrails-hitl-approve"}}

result = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Send an email to team@company.com with subject 'Q4 Results' "
                "and body 'Revenue grew 12%.'"
            ),
        }]
    },
    config=config_approve,
)

if "__interrupt__" in result:
    print("Paused — awaiting approval:", result["__interrupt__"])
    result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_approve,
    )

print("=== After approval ===")
print(result["messages"][-1].content)

In [ ]:
# Reject flow — delete_records requires human sign-off
config_reject = {"configurable": {"thread_id": "guardrails-hitl-reject"}}

hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Delete all records from users where active=false",
        }]
    },
    config=config_reject,
)

rejected = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky — needs DBA review"}]}),
    config=config_reject,
)
print("=== After rejection ===")
print(rejected["messages"][-1].content)

## 5. Custom guardrail: `before_agent` (input filter)

Subclass `AgentMiddleware` and implement `before_agent`. Use `@hook_config(can_jump_to=["end"])` to short-circuit the graph with a safe refusal — **zero LLM cost** for blocked requests.

**Best for:** keyword filters, auth checks, rate limits, category blocking.

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.tools import tool
from langgraph.runtime import Runtime


class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic input guardrail — blocks banned keywords before any LLM call."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()
        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked — keyword: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        ),
                    }],
                    "jump_to": "end",
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


filtered_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)
print("Content-filter agent created.")

In [ ]:
# Safe request passes through
safe = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe:", safe["messages"][-1].content[:120], "...")

# Unsafe request blocked at before_agent
unsafe = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Blocked:", unsafe["messages"][-1].content)

## 6. Custom guardrail: `after_agent` (output safety)

Use `after_agent` to inspect or rewrite the **final** AI message before it reaches the user.

**Best for:** model-based output judges, compliance scans, disclaimer injection, catching PII that slipped through.

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime


class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based output guardrail — lightweight judge on final response."""

    def __init__(self):
        super().__init__()
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = (
            "Evaluate if this AI response is safe and appropriate. "
            "Reply with only 'SAFE' or 'UNSAFE'.\n\n"
            f"Response:\n{last_message.content}"
        )
        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("Output flagged UNSAFE — replacing with fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )
        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="gpt-4o-mini",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)
print("Output-safety agent created.")

In [ ]:
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print(result["messages"][-1].content)

## 7. Layered / combined guardrails

Stack middleware in `middleware=[...]`. They run **in list order** — defense in depth.

```
User input
  → [1] ContentFilterMiddleware      (before_agent)
  → [2] PIIMiddleware (input)        (before_model)
  → [3] HumanInTheLoopMiddleware     (wrap_tool_call)
  → [4] PIIMiddleware (output)       (before_model / output path)
  → [5] SafetyGuardrailMiddleware    (after_agent)
User response
```

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_tool_layered(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"


@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"


production_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_tool_layered, send_email_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool_layered": False}
        ),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)
print("Production agent with 5-layer guardrails ready.")

## 8. Real-world use case: healthcare chatbot

Combines multiple guardrail types for a regulated domain:

| Layer | Middleware | Purpose |
|-------|------------|---------|
| 1 | `HealthcareSafetyFilter` | Block off-topic / harmful requests |
| 2 | `PIIMiddleware` | Redact patient email, mask cards |
| 3 | `HumanInTheLoopMiddleware` | Approve `book_appointment` |
| 4 | `MedicalOutputValidator` | Append medical disclaimer |

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime


class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = [
        "drug synthesis", "synthesize", "self-harm", "suicide method", "weapon", "hack",
    ]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None
        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with medical "
                            "questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        ),
                    }],
                    "jump_to": "end",
                }
        return None


class MedicalOutputValidator(AgentMiddleware):
    """Append a medical disclaimer to every response."""

    DISCLAIMER = (
        "\n\n*This is general health information, not medical advice. "
        "Please consult a qualified healthcare professional.*"
    )

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER
        return None


@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."


@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"


@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


healthcare_bot = create_agent(
    model="gpt-4o-mini",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        HealthcareSafetyFilter(),
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search symptoms, medication info, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    ),
)
print("Healthcare chatbot created.")

In [ ]:
from langgraph.types import Command

config_health = {"configurable": {"thread_id": "healthcare-demo-1"}}

# Test 1: Safe medical query (symptom search + disclaimer)
r1 = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_health,
)
print("=== Symptoms ===")
print(r1["messages"][-1].content[:300], "...\n")

# Test 2: PII redacted on input
r2 = healthcare_bot.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "My email is patient123@gmail.com. What can I take for a headache?",
        }]
    },
    config=config_health,
)
print("=== PII (check first message) ===")
print("Input seen by model:", r2["messages"][0].content[:80], "...\n")

# Test 3: Harmful request blocked at before_agent
r3 = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]},
    config=config_health,
)
print("=== Blocked ===")
print(r3["messages"][-1].content[:200], "...")

In [ ]:
# Test 4: Appointment booking — human approval required
config_appt = {"configurable": {"thread_id": "healthcare-appt-1"}}

result = healthcare_bot.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Book an appointment for Jane Doe with Dr. Sharma on March 15",
        }]
    },
    config=config_appt,
)

if "__interrupt__" in result:
    print("Paused for appointment approval")
    result = healthcare_bot.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_appt,
    )

print("=== After booking flow ===")
print(result["messages"][-1].content)

## 9. Notebook summary

### Guardrail map

| Guardrail | Hook | When it runs | Best for |
|-----------|------|--------------|----------|
| `PIIMiddleware` | `before_model` | Around model I/O | Privacy, compliance |
| `HumanInTheLoopMiddleware` | `wrap_tool_call` | Before sensitive tools | High-stakes actions |
| `ContentFilterMiddleware` | `before_agent` | Start of invoke | Cheap input blocking |
| `SafetyGuardrailMiddleware` | `after_agent` | End of invoke | Output safety judge |
| Custom `AgentMiddleware` | Any hook | Your choice | Business rules |

### Key takeaways

1. **Guardrails = middleware** — pass to `create_agent(..., middleware=[...])`
2. **Layer them** — deterministic first, model-based second, human approval for irreversible actions
3. **PII strategies** — `redact` / `mask` / `hash` / `block` per field sensitivity
4. **HITL needs memory** — `checkpointer` + same `thread_id` for `Command(resume=...)`
5. **`jump_to: "end"`** — short-circuit graph with a safe message (no LLM spend)

### Quick reference

```python
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o-mini",
    tools=[...],
    middleware=[
        ContentFilterMiddleware(banned_keywords=[...]),           # before_agent
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        HumanInTheLoopMiddleware(interrupt_on={"risky_tool": True}),
        SafetyGuardrailMiddleware(),                                # after_agent
    ],
    checkpointer=InMemorySaver(),  # required for HITL
)
```

### Tutorial series

| Notebook | Topic |
|----------|-------|
| [6_middleware](6_middleware.ipynb) | Hooks, summarization, HITL deep dive |
| **7_guardrails** (this) | PII, input/output safety, layered protection |

**Guardrails diagram:** [reference/guardrails-flow.png](reference/guardrails-flow.png) · **Middleware hooks:** [reference/middleware-design.png](reference/middleware-design.png)